# US Accidents (2016–2023) — EDA, Cleaning, Visualization & Severity/State PredictionThis notebook merges the shared exploration/cleaning work with the two branches that grew out of it:1. **Visualization branch** — dashboard-style charts.2. **Modeling branch** — feature engineering + LightGBM models for `Severity` and `State`.> Full dataset (`US_Accidents_March23.csv`) is **not included** in this repo (too large for GitHub). See `data/README.md` for the Kaggle source link. The Tableau dashboard was built on `sample2.csv`, a 1,000,000-row random sample generated in the *Sampling* section below.

## 1. Load Data & Initial Exploration

In [ ]:
import pandas as pd

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("data/US_Accidents_March23.csv")  # update path locally; file not committed to git

In [ ]:
df.shape 

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

## 2. Datetime Parsing & Missing-Value Overview

In [ ]:
df["Start_Time"] = pd.to_datetime(df["Start_Time"], format="mixed", errors="coerce")
df["End_Time"] = pd.to_datetime(df["End_Time"], format="mixed", errors="coerce")# هنا استخدمنا ابفرومات ميكسيد عشان بانداز يتعامل مع اكالفورم المختلف للتاريخ 
df["Weather_Timestamp"] = pd.to_datetime(df["Weather_Timestamp"], format="mixed", errors="coerce")


print(df[["Start_Time", "End_Time", "Weather_Timestamp"]].dtypes)

In [ ]:
print(df[["Start_Time", "End_Time", "Weather_Timestamp"]].isna().sum())

In [ ]:
missing = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (df.isnull().sum() / len(df)) * 100
})

missing = missing[missing["Missing_Count"] > 0].sort_values(
    "Missing_Percentage",
    ascending=False
)

print(missing)

In [ ]:
df["Year"] = df["Start_Time"].dt.year
df["Month"] = df["Start_Time"].dt.month
df["Day"] = df["Start_Time"].dt.day
df["Day_of_Week"] = df["Start_Time"].dt.day_name()
df["Hour"] = df["Start_Time"].dt.hour


df["Is_Weekend"] = df["Start_Time"].dt.dayofweek >= 5


# عشان نجاوب علي اي سنه و اسبوع و شهر في حوادث اكتر و هل الويكند بياثر ؟



## 3. Severity Distribution (Raw)

In [ ]:
print(df["Severity"].value_counts().sort_index())
print(df["Severity"].value_counts(normalize=True).sort_index() * 100)
print(df["Year"].value_counts().sort_index())

In [ ]:
df["Severity"].value_counts().sort_index()

In [ ]:
df["Severity"].value_counts(normalize=True).sort_index() * 100

#النورماليز بيجيب النئبه المؤيه بس محتاجين نضربها في 100

## 4. Time Analysis (Year / Month / Day of Week / Day-Type / Hour / Day-Night)

In [ ]:

df["Year"].value_counts().sort_index()
# عملت عمود جدبد فيه السنين 

In [ ]:
df.groupby("Year")["Start_Time"].agg(["min", "max"])




In [ ]:
yearly_df = df[
    (df["Year"] >= 2017) &
    (df["Year"] <= 2022)
] #هنا عملت داتا فيها السنوات الكامله بس عشان 2016 و 2023 مش كاملين

In [ ]:
df["Month"].value_counts().sort_index()
# هنا طبعا بشوف كل شهر حصل فيه قد ايه 

In [ ]:
df["Month"].value_counts(normalize=True).sort_index() * 100
# و هنا بشوف النسبه 

In [ ]:
month_counts = df["Month"].value_counts().sort_index()

print(month_counts)
# و هنا بشوف عملتىكل شهر في قد ايه حوادث

In [ ]:
month_percent = df["Month"].value_counts(normalize=True).sort_index() * 100

print(month_percent)

In [ ]:
monthly_severity = pd.crosstab(
    df["Month"],
    df["Severity"],
    normalize="index"
) * 100

print(monthly_severity.round(2))
# هنا نسبة كل قيمه من خطورة الحادق ل كل شهر 

In [ ]:
days_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]# ده هنا بنعمل انديكس عشان نستخدمه بنفس الترتيب ده

day_counts = df["Day_of_Week"].value_counts().reindex(days_order) # كده كده لو معملنا كده هيطبع ايام عادي بس بترتيب الاكبر للاصغر و احنا عايزين ترتيب ايام الاسبوع 

print(day_counts)

In [ ]:
day_percent = (
    df["Day_of_Week"]
    .value_counts(normalize=True)
    .reindex(days_order)
    * 100
)

print(day_percent.round(2))

In [ ]:
daily_severity = pd.crosstab(
    df["Day_of_Week"],
    df["Severity"],
    normalize="index"
).reindex(days_order) * 100

print(daily_severity.round(2))

In [ ]:
df["Day_Type"] = df["Is_Weekend"].map({
    False: "Weekday",
    True: "Weekend"
})

In [ ]:
print(df["Day_Type"].value_counts())

In [ ]:
print(
    df["Day_Type"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

In [ ]:
weekend_severity = pd.crosstab(
    df["Day_Type"],
    df["Severity"],
    normalize="index"
) * 100

print(weekend_severity.round(2))

In [ ]:


hour_counts = df["Hour"].value_counts().sort_index()

print(hour_counts)

In [ ]:
hour_percent = (
    df["Hour"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

print(hour_percent.round(2))

In [ ]:
hour_severity = pd.crosstab(
    df["Hour"],
    df["Severity"],
    normalize="index"
) * 100

print(hour_severity.round(2))

In [ ]:
df["Sunrise_Sunset"].value_counts(dropna=False)

In [ ]:
day_night_severity = pd.crosstab(
    df["Sunrise_Sunset"],
    df["Severity"],
    normalize="index"
) * 100
# طيعا هتسال لي مفيش nan  في الاوتبوت عشان الcrosstab بيستبعد ال nan
print(day_night_severity.round(2))

In [ ]:
full_year_df = df[
    df["Year"].between(2017, 2022)
].copy()

In [ ]:
year_counts = (
    full_year_df["Year"]
    .value_counts()
    .sort_index()
)

print(year_counts)

In [ ]:
year_severity = pd.crosstab(
    full_year_df["Year"],
    full_year_df["Severity"],
    normalize="index"
) * 100

print(year_severity.round(2))

## 5. Data Cleaning (first pass — missing value imputation)

In [ ]:
df_clean = df.copy()



In [ ]:
numeric_cols = [
    "Temperature(F)",
    "Humidity(%)",
    "Pressure(in)",
    "Visibility(mi)",
    "Wind_Speed(mph)",
    "Wind_Chill(F)"
]

for col in numeric_cols:
    df_clean[col] = df_clean[col].fillna(
        df_clean[col].median()
    )

    df_clean["Sunrise_Sunset"] = df_clean["Sunrise_Sunset"].fillna("Unknown")

df_clean["City"] = df_clean["City"].fillna(
    df_clean["City"].mode()[0]
)

df_clean["Zipcode"] = df_clean["Zipcode"].fillna("Unknown")

df_clean["Timezone"] = df_clean["Timezone"].fillna("Unknown")

In [ ]:
print(df_clean.isnull().sum().sort_values(ascending=False))

In [ ]:
df_clean["Wind_Direction"] = df_clean["Wind_Direction"].fillna(
    df_clean["Wind_Direction"].mode()[0]  # عوضنا هنا بالمود 
)

In [ ]:
df_clean["Weather_Condition"] = df_clean["Weather_Condition"].fillna(
    df_clean["Weather_Condition"].mode()[0]
)

In [ ]:
df_clean["Civil_Twilight"] = df_clean["Civil_Twilight"].fillna("Unknown")
df_clean["Nautical_Twilight"] = df_clean["Nautical_Twilight"].fillna("Unknown")
df_clean["Astronomical_Twilight"] = df_clean["Astronomical_Twilight"].fillna("Unknown")
df_clean["Airport_Code"] = df_clean["Airport_Code"].fillna("Unknown")
df_clean["Street"] = df_clean["Street"].fillna("Unknown")
df_clean["Description"] = df_clean["Description"].fillna("Unknown")

In [ ]:
missing_final = (
    df_clean.isnull()
    .sum()
    .sort_values(ascending=False)
)

print(missing_final[missing_final > 0])

In [ ]:
print("Original shape:", df.shape)
print("Cleaned shape:", df_clean.shape)

In [ ]:
print(df_clean["Severity"].isnull().sum())

In [ ]:
print("Original:", df.shape)
print("Cleaned:", df_clean.shape)

print("\nRemaining Missing Values:")
print(df_clean.isnull().sum().sort_values(ascending=False).head(10))

print("\nDuplicate Rows:")
print(df_clean.duplicated().sum())

print("\nSeverity Missing:")
print(df_clean["Severity"].isnull().sum())

In [ ]:
# لغاية هنا عملنا كلينينج لبقيت الفاليوز 

## 6. Location Analysis

In [ ]:
state_counts = df_clean["State"].value_counts()

print(state_counts.head(20))

In [ ]:
state_percent = (
    df_clean["State"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print(state_percent.head(20))

In [ ]:
state_severity = pd.crosstab(
    df_clean["State"],
    df_clean["Severity"],
    normalize="index"
) * 100

print(state_severity.round(2).head(20))

In [ ]:
city_counts = df_clean["City"].value_counts()

print(city_counts.head(20))

In [ ]:
city_percent = (
    df_clean["City"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print(city_percent.head(20))

In [ ]:
street_counts = df_clean["Street"].value_counts()

print(street_counts.head(20))

## 7. Weather Analysis

In [ ]:
weather_counts = (
    df_clean["Weather_Condition"]
    .value_counts()
)

print(weather_counts.head(20))

In [ ]:
weather_severity = pd.crosstab(
    df_clean["Weather_Condition"],
    df_clean["Severity"],
    normalize="index"
) * 100

print(weather_severity.round(2).head(20))

In [ ]:
temp_severity = df_clean.groupby("Severity")["Temperature(F)"].agg(
    ["count", "mean", "median", "min", "max"]
).round(2)

print(temp_severity)

In [ ]:
visibility_severity = df_clean.groupby("Severity")["Visibility(mi)"].agg(
    ["count", "mean", "median", "min", "max"]
).round(2)

print(visibility_severity)

In [ ]:
humidity_severity = df_clean.groupby("Severity")["Humidity(%)"].agg(
    ["count", "mean", "median", "min", "max"]
).round(2)

print(humidity_severity)

In [ ]:
precip_severity = df_clean.groupby("Severity")["Precipitation(in)"].agg(
    ["count", "mean", "median", "min", "max"]
).round(3)

print(precip_severity)


In [ ]:
wind_severity = df_clean.groupby("Severity")["Wind_Speed(mph)"].agg(
    ["count", "mean", "median", "min", "max"]
).round(2)

print(wind_severity)

## 8. Road Features vs. Severity

In [ ]:
road_features = [
    "Amenity",
    "Bump",
    "Crossing",
    "Give_Way",
    "Junction",
    "No_Exit",
    "Railway",
    "Roundabout",
    "Station",
    "Stop",
    "Traffic_Calming",
    "Traffic_Signal",
    "Turning_Loop"
]

road_counts = df_clean[road_features].sum().sort_values(ascending=False)

print(road_counts)

In [ ]:
road_severity = {}

for feature in road_features:
    road_severity[feature] = (
        df_clean.groupby(feature)["Severity"]
        .count()
    )

print(road_severity)

In [ ]:
road_severity_34 = {}

for feature in road_features:
    subset = df_clean[df_clean[feature] == True]
    
    road_severity_34[feature] = (
        subset["Severity"].isin([3, 4]).mean() * 100
    )

road_severity_34 = pd.Series(road_severity_34).sort_values(ascending=False)

print(road_severity_34.round(2))

## 9. Dashboard VisualizationsCharts used to sanity-check the numbers behind the Tableau dashboard (built separately on `sample2.csv`).

In [ ]:
# Modern color palette
COLORS = [
    "#2563EB",  # Blue
    "#7C3AED",  # Purple
    "#EC4899",  # Pink
    "#F59E0B",  # Amber
    "#10B981",  # Emerald
    "#06B6D4",  # Cyan
    "#EF4444",  # Red
    "#8B5CF6",  # Violet
]

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["font.size"] = 11

In [ ]:
year_counts = df_clean["Year"].value_counts().sort_index()

plt.figure(figsize=(11, 6))

plt.plot(
    year_counts.index,
    year_counts.values,
    marker="o",
    linewidth=3,
    color="#2563EB"
)

plt.title("Recorded Accidents by Year", fontweight="bold")
plt.xlabel("Year")
plt.ylabel("Number of Accidents")

plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
month_counts = df_clean["Month"].value_counts().sort_index()

month_names = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
]

plt.figure(figsize=(12, 6))

bars = plt.bar(
    month_names,
    month_counts.values,
    color="#7C3AED"
)

plt.title("Recorded Accidents by Month", fontweight="bold")
plt.xlabel("Month")
plt.ylabel("Number of Accidents")

plt.grid(axis="y", alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

day_counts = (
    df_clean["Day_of_Week"]
    .value_counts()
    .reindex(day_order)
)

plt.figure(figsize=(11, 6))

bars = plt.bar(
    day_counts.index,
    day_counts.values,
    color="#10B981"
)

plt.title("Recorded Accidents by Day of Week", fontweight="bold")
plt.xlabel("Day")
plt.ylabel("Number of Accidents")

plt.xticks(rotation=20)
plt.grid(axis="y", alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
hour_counts = df_clean["Hour"].value_counts().sort_index()

plt.figure(figsize=(12, 6))

plt.plot(
    hour_counts.index,
    hour_counts.values,
    marker="o",
    linewidth=2.5,
    color="#EC4899"
)

plt.title("Recorded Accidents by Hour", fontweight="bold")
plt.xlabel("Hour of Day")
plt.ylabel("Number of Accidents")

plt.xticks(range(0, 24))
plt.grid(axis="both", alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
state_counts = (
    df_clean["State"]
    .value_counts()
    .head(10)
    .sort_values()
)

plt.figure(figsize=(10, 7))

plt.barh(
    state_counts.index,
    state_counts.values,
    color="#F59E0B"
)

plt.title("Top 10 States by Recorded Accidents", fontweight="bold")
plt.xlabel("Number of Accidents")
plt.ylabel("State")

plt.grid(axis="x", alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
city_counts = (
    df_clean["City"]
    .value_counts()
    .head(10)
    .sort_values()
)

plt.figure(figsize=(10, 7))

plt.barh(
    city_counts.index,
    city_counts.values,
    color="#06B6D4"
)

plt.title("Top 10 Cities by Recorded Accidents", fontweight="bold")
plt.xlabel("Number of Accidents")
plt.ylabel("City")

plt.grid(axis="x", alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
weather_counts = (
    df_clean["Weather_Condition"]
    .value_counts()
    .head(10)
    .sort_values()
)

plt.figure(figsize=(11, 7))

plt.barh(
    weather_counts.index,
    weather_counts.values,
    color="#8B5CF6"
)

plt.title(
    "Top 10 Weather Conditions Associated with Recorded Accidents",
    fontweight="bold"
)

plt.xlabel("Number of Accidents")
plt.ylabel("Weather Condition")

plt.grid(axis="x", alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
day_night_severity = pd.crosstab(
    df_clean["Sunrise_Sunset"],
    df_clean["Severity"],
    normalize="index"
) * 100

day_night_severity = day_night_severity.reindex(
    ["Day", "Night"]
)

ax = day_night_severity.plot(
    kind="bar",
    stacked=True,
    figsize=(10, 6),
    color=["#2563EB", "#10B981", "#F59E0B", "#EF4444"]
)

plt.title(
    "Accident Severity Distribution: Day vs Night",
    fontweight="bold"
)

plt.xlabel("Time of Day")
plt.ylabel("Percentage of Accidents")

plt.legend(
    title="Severity",
    labels=["Severity 1", "Severity 2", "Severity 3", "Severity 4"]
)

plt.xticks(rotation=0)
plt.ylim(0, 100)

plt.tight_layout()
plt.show()

In [ ]:
road_features = [
    "Amenity",
    "Bump",
    "Crossing",
    "Give_Way",
    "Junction",
    "No_Exit",
    "Railway",
    "Roundabout",
    "Station",
    "Stop",
    "Traffic_Calming",
    "Traffic_Signal",
    "Turning_Loop"
]

severe_percent = {}

for feature in road_features:
    subset = df_clean[df_clean[feature] == True]

    if len(subset) > 0:
        severe_percent[feature] = (
            subset["Severity"].isin([3, 4]).mean() * 100
        )

severe_percent = (
    pd.Series(severe_percent)
    .sort_values()
)

plt.figure(figsize=(11, 8))

plt.barh(
    severe_percent.index,
    severe_percent.values,
    color="#EF4444"
)

plt.axvline(
    19.46,
    linestyle="--",
    linewidth=2,
    color="#111827",
    label="Overall Severe Accident Rate"
)

plt.title(
    "Severe Accident Rate by Road Feature",
    fontweight="bold"
)

plt.xlabel("Severity 3 & 4 (%)")
plt.ylabel("Road Feature")

plt.legend()

plt.grid(axis="x", alpha=0.2)

plt.tight_layout()
plt.show()

## 10. Advanced Cleaning (for modeling)A second, stricter cleaning pass — drop high-missingness columns, drop remaining incomplete rows, drop constant/low-value columns — to prepare a modeling-ready dataframe.

In [ ]:
print("Total columns:", len(df.columns))
print(df.columns.tolist())


In [ ]:
# 2. More detailed view: column name, data type, non-null count, missing count
summary = pd.DataFrame({
    "Column": df.columns,
    "Dtype": df.dtypes.values,
    "Non-Null Count": df.notnull().sum().values,
    "Missing Count": df.isnull().sum().values,
    "Missing %": (df.isnull().mean() * 100).round(2).values
})
print(summary.to_string(index=False))

In [ ]:
# Step 1: Drop columns with very high missingness (not worth imputing/keeping)
high_missing_cols = ["End_Lat", "End_Lng"]  # 25%+ missing
df = df.drop(columns=high_missing_cols)

# Drop rows missing ANY of these columns (including previously "high missing" ones)
cols_to_check = [
    "Wind_Chill(F)",
    "Description", "Street", "City", "Zipcode", "Timezone",
    "Airport_Code", "Weather_Timestamp", "Temperature(F)",
    "Humidity(%)", "Pressure(in)"
]

df = df.dropna(subset=cols_to_check)

print("Remaining rows:", len(df))

In [ ]:
categorical_cols = ["City", "Timezone", "Weather_Condition", "Wind_Direction","Sunrise_Sunset", "Civil_Twilight", "Nautical_Twilight", "Astronomical_Twilight"]
numeric_cols = ["Temperature(F)", "Wind_Chill(F)", "Humidity(%)", "Pressure(in)", 
                 "Visibility(mi)", "Wind_Speed(mph)", "Precipitation(in)"]

# Drop low-value location metadata instead of imputing
df = df.drop(columns=["Street", "Zipcode", "Airport_Code"])

# Add missing-value flags before filling
for col in numeric_cols:
    df[col + "_was_missing"] = df[col].isnull().astype(int)

# Group-based median imputation (weather is regional/seasonal)
for col in numeric_cols:
    df[col] = df.groupby(["State", "Month"])[col].transform(lambda x: x.fillna(x.median()))
    df[col] = df[col].fillna(df[col].median())  # fallback for empty groups

# Fill categoricals with "Unknown"
for col in categorical_cols:
    df[col] = df[col].fillna("Unknown")

In [ ]:
print(df.isnull().sum().sum())  # should be 0 (or close to it)



In [ ]:
print(df["Country"].head(20))
print(df["Country"].unique())        # see all distinct values
print(df["Country"].value_counts())  # see distribution

In [ ]:
if df["Country"].nunique() == 1:
    df = df.drop(columns=["Country"])
    print("Dropped Country — constant column")

## 11. Feature Selection, Outlier Check & Sampling`sample1.csv` / `sample2.csv` / `sample3.csv` are three 1,000,000-row random samples generated here — **`sample2.csv` is the file the Tableau dashboard is built on.**

In [ ]:
target = "Severity"

selected_cols = [
    "Source", "Start_Lat", "Start_Lng", "State", "Timezone",
    "Temperature(F)", "Wind_Chill(F)", "Humidity(%)", "Pressure(in)",
    "Visibility(mi)", "Wind_Direction", "Wind_Speed(mph)", "Precipitation(in)",
    "Weather_Condition",
    "Amenity", "Bump", "Crossing", "Give_Way", "Junction", "No_Exit",
    "Railway", "Roundabout", "Station", "Stop", "Traffic_Calming",
    "Traffic_Signal", "Turning_Loop",
    "Sunrise_Sunset", "Civil_Twilight", "Nautical_Twilight", "Astronomical_Twilight",
    "Year", "Month", "Day", "Hour", "Day_of_Week", "Is_Weekend", "Day_Type"
]

X = df[selected_cols].copy()
y = df[target]

print("Features shape:", X.shape)
print("Target distribution:\n", y.value_counts(normalize=True))

In [ ]:
print(X.dtypes)
print("\nUnique value counts per column:")
for col in X.columns:
    print(col, X[col].nunique())

In [ ]:
for col in ["State", "Timezone", "Weather_Condition", "Wind_Direction"]:
    print(col, X[col].nunique())

In [ ]:
import matplotlib.pyplot as plt
numeric_cols = ["Temperature(F)", "Wind_Chill(F)", "Humidity(%)", "Pressure(in)",
                 "Visibility(mi)", "Wind_Speed(mph)", "Precipitation(in)"]

# Sample for faster plotting (visualization only, not modifying real data)
sample = df[numeric_cols].sample(200_000, random_state=42)

fig, axes = plt.subplots(4, 2, figsize=(14, 18))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].boxplot(sample[col].dropna(), vert=True)
    axes[i].set_title(col)
    axes[i].set_ylabel(col)

# Hide the unused 8th subplot
axes[-1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
sample_size = 1000000

sample1 = df.sample(n=sample_size, random_state=1)
sample2 = df.sample(n=sample_size, random_state=2)
sample3 = df.sample(n=sample_size, random_state=3)

# Save each to Excel (or CSV — faster and Excel opens CSVs fine too)
sample1.to_csv("sample1.csv", index=False)
sample2.to_csv("sample2.csv", index=False)
sample3.to_csv("sample3.csv", index=False)

print("Sample 1 shape:", sample1.shape)
print("Sample 2 shape:", sample2.shape)
print("Sample 3 shape:", sample3.shape)

In [ ]:
# High-cardinality columns → frequency encode
high_card_cols = ["State", "Timezone", "Weather_Condition", "Wind_Direction"]

for col in high_card_cols:
    freq = X[col].value_counts()
    X[col] = X[col].map(freq)

# Low-cardinality categoricals + booleans → one-hot encode
X = pd.get_dummies(X, drop_first=True)

print("Final feature count:", X.shape[1])

## 12. Severity Prediction Model (LightGBM)Iterating on `class_weight` to improve minority-class (severity 4) recall while keeping accuracy reasonable.

In [ ]:
# predict severity of the accident
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

In [ ]:
# first trail
from sklearn.metrics import classification_report, f1_score
import lightgbm as lgb
model = lgb.LGBMClassifier(
    objective="multiclass",
    class_weight="balanced",
    n_estimators=300,
    learning_rate=0.05,
    random_state=42
)

model.fit(X_train, y_train)
y_pred_base = model.predict(X_test)
print(classification_report(y_test, y_pred_base))

In [ ]:

# secand trail
model_baseline = lgb.LGBMClassifier(
    objective="multiclass",
    n_estimators=300,
    learning_rate=0.05,
    random_state=42
)

model_baseline.fit(X_train, y_train)
y_pred_base = model_baseline.predict(X_test)

print(classification_report(y_test, y_pred_base))

In [ ]:
# third trail # v3 ({1:3, 2:1, 3:2, 4:6})	            0.63 ✅                      0.87	                    0.36
model_v3 = lgb.LGBMClassifier(
    objective="multiclass",
    class_weight={1: 3, 2: 1, 3: 2, 4: 6},   # gentle boost, not full "balanced"
    n_estimators=300,
    learning_rate=0.05,
    random_state=42
)
cat_cols = X_train.select_dtypes(include=["str", "object"]).columns.tolist()
for col in cat_cols:
    X_train[col] = X_train[col].astype("category")
    X_test[col] = X_test[col].astype("category")

model_v3.fit(X_train, y_train, categorical_feature=cat_cols)

y_pred_v3 = model_v3.predict(X_test)

from sklearn.metrics import classification_report, f1_score
print(classification_report(y_test, y_pred_v3))
print("Macro F1:", f1_score(y_test, y_pred_v3, average="macro"))



In [ ]:
# three model output (best so far)
# Model                              	Macro F1	                 Accuracy                 	Class 4 Recall
# Balanced (class_weight="balanced")	0.51	                      0.69	                    0.73 (but precision only 0.11)
# No weighting (baseline)	            0.58	                     0.89	                    0.08
# v3 ({1:3, 2:1, 3:2, 4:6})	            0.63 ✅                      0.87	                    0.36

In [ ]:

# fourth trial 
import lightgbm as lgb
from sklearn.metrics import classification_report, f1_score

weight_options = [
    {1: 3, 2: 1, 3: 2, 4: 6},    # your current best (v3)
    {1: 4, 2: 1, 3: 2, 4: 8},
    {1: 3, 2: 1, 3: 3, 4: 8},
    {1: 4, 2: 1, 3: 3, 4: 10},
    {1: 2, 2: 1, 3: 2, 4: 6},
]

results = []
for w in weight_options:
    m = lgb.LGBMClassifier(
        objective="multiclass",
        class_weight=w,
        n_estimators=300,
        learning_rate=0.05,
        random_state=42
    )
    m.fit(X_train, y_train)
    preds = m.predict(X_test)
    score = f1_score(y_test, preds, average="macro")
    results.append((w, score))
    print(w, "-> Macro F1:", round(score, 4))

best_weights, best_score = max(results, key=lambda x: x[1])
print("\nBest weights:", best_weights, "Macro F1:", best_score)

print(classification_report(y_test, y_pred_base))

In [ ]:
# searching for best possible value 
weight_options_v2 = [
    {1: 1, 2: 1, 3: 2, 4: 6},
    {1: 2, 2: 1, 3: 2, 4: 5},
    {1: 2, 2: 1, 3: 2, 4: 7},
    {1: 2, 2: 1, 3: 1, 4: 6},
    {1: 1, 2: 1, 3: 2, 4: 7},
]

results_v2 = []
for w in weight_options_v2:
    m = lgb.LGBMClassifier(
        objective="multiclass",
        class_weight=w,
        n_estimators=300,
        learning_rate=0.05,
        random_state=42
    )
    m.fit(X_train, y_train)
    preds = m.predict(X_test)
    score = f1_score(y_test, preds, average="macro")
    results_v2.append((w, score))
    print(w, "-> Macro F1:", round(score, 4))

best_v2 = max(results_v2, key=lambda x: x[1])
print("\nBest v2:", best_v2)

In [ ]:
model_final = lgb.LGBMClassifier(
    objective="multiclass",
    class_weight={1: 2, 2: 1, 3: 2, 4: 6},
    n_estimators=300,
    learning_rate=0.05,
    random_state=42
)
model_final.fit(X_train, y_train)
y_pred_final = model_final.predict(X_test)

print(classification_report(y_test, y_pred_final))
print("Macro F1:", f1_score(y_test, y_pred_final, average="macro"))

print(classification_report(y_test, y_pred_base))

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

y_pred = model_final.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

In [ ]:
import numpy as np
models = ["Balanced\nWeights", "No\nWeights", "Tuned\nWeights (Final)"]
accuracy = [0.69, 0.89, 0.87]
macro_f1 = [0.51, 0.58, 0.63]

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 6))
ax.bar(x - width/2, accuracy, width,color='#ABD2FA',label="Accuracy")
ax.bar(x + width/2, macro_f1, width,color='#7692FF',label="Macro F1")

ax.set_ylabel("Score")
ax.set_title("Model Iteration Comparison")
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()
ax.set_ylim(0, 1)

for i, v in enumerate(accuracy):
    ax.text(i - width/2, v + 0.02, f"{v:.2f}", ha="center")
for i, v in enumerate(macro_f1):
    ax.text(i + width/2, v + 0.02, f"{v:.2f}", ha="center")

plt.tight_layout()
plt.show()

In [ ]:
print(df["Timezone"].unique())
print(df["State"].nunique())

## 13. State Prediction Model (LightGBM)

In [ ]:
target = "State"

selected_cols = [
    "Temperature(F)", "Wind_Chill(F)", "Humidity(%)", "Pressure(in)",
    "Visibility(mi)", "Wind_Direction", "Wind_Speed(mph)", "Precipitation(in)",
    "Weather_Condition",
    "Amenity", "Bump", "Crossing", "Give_Way", "Junction", "No_Exit",
    "Railway", "Roundabout", "Station", "Stop", "Traffic_Calming",
    "Traffic_Signal", "Turning_Loop",
    "Sunrise_Sunset", "Civil_Twilight", "Nautical_Twilight", "Astronomical_Twilight",
    "Year", "Month", "Day", "Hour", "Day_of_Week", "Is_Weekend", "Day_Type"
]


X = df[selected_cols].copy()
y = df[target]

print(X.columns.tolist()) 

In [ ]:
from sklearn.model_selection import train_test_split

high_card_cols = ["Weather_Condition", "Wind_Direction"]
for col in high_card_cols:
    freq = X[col].value_counts()
    X[col] = X[col].map(freq)

# One-hot encode only the LOW-cardinality remaining categoricals
low_card_cols = X.select_dtypes(include=["str", "object"]).columns.tolist()
X = pd.get_dummies(X, columns=low_card_cols, drop_first=True)

print("Feature count:", X.shape[1]) 

In [ ]:


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# long loaded data 89min
import lightgbm as lgb
from sklearn.metrics import classification_report, f1_score
model_state = lgb.LGBMClassifier(
    objective="multiclass",
    n_estimators=300,
    learning_rate=0.05,
    random_state=42
)

model_state.fit(X_train, y_train)
y_pred_state = model_state.predict(X_test)

print(classification_report(y_test, y_pred_state))
print("Macro F1:", f1_score(y_test, y_pred_state, average="macro"))

In [ ]:
sample_df = df.sample(500_000, random_state=42)

selected_cols = [
    "Temperature(F)", "Wind_Chill(F)", "Humidity(%)", "Pressure(in)",
    "Visibility(mi)", "Wind_Direction", "Wind_Speed(mph)", "Precipitation(in)",
    "Weather_Condition",
    "Amenity", "Bump", "Crossing", "Give_Way", "Junction", "No_Exit",
    "Railway", "Roundabout", "Station", "Stop", "Traffic_Calming",
    "Traffic_Signal", "Turning_Loop",
    "Sunrise_Sunset", "Civil_Twilight", "Nautical_Twilight", "Astronomical_Twilight",
    "Year", "Month", "Day", "Hour", "Day_of_Week", "Is_Weekend", "Day_Type"
]

X = sample_df[selected_cols].copy()
y = sample_df["State"]

cat_cols = X.select_dtypes(include=["str", "object", "bool"]).columns.tolist()
for col in cat_cols:
    X[col] = X[col].astype("category")

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
import lightgbm as lgb
model_state = lgb.LGBMClassifier(
    objective="multiclass",
    class_weight="balanced",
    n_estimators=300,
    learning_rate=0.05,
    random_state=42
)
model_state.fit(X_train, y_train, categorical_feature=cat_cols)

In [ ]:
y_pred_state = model_state.predict(X_test)

from sklearn.metrics import classification_report, f1_score
print(classification_report(y_test, y_pred_state))
print("Macro F1:", f1_score(y_test, y_pred_state, average="macro"))

In [ ]:
from sklearn.metrics import classification_report

# Get the report as a dict instead of printed text — much easier to work with
report = classification_report(y_test, y_pred_state, output_dict=True)

# Convert to a DataFrame and drop the summary rows (accuracy, macro avg, weighted avg)
report_df = pd.DataFrame(report).T
report_df = report_df.drop(index=["accuracy", "macro avg", "weighted avg"], errors="ignore")

# Optional: sort by support (sample size) so you can see if score quality tracks sample size
report_df = report_df.sort_values("support", ascending=False)

# Plot precision, recall, f1-score as grouped bars
ax = report_df[["precision", "recall", "f1-score"]].plot(
    kind="bar",
    figsize=(16, 6),
    width=0.8
)

ax.set_title("Classification Report by State")
ax.set_xlabel("State")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.legend(loc="upper right")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
region_map = {
    "CA": "West", "OR": "West", "WA": "West", "NV": "West", "AZ": "Southwest", "NM": "Southwest", "UT": "West", "CO": "West", "ID": "West", "MT": "West", "WY": "West",
    "TX": "South", "OK": "South", "AR": "South", "LA": "South", "MS": "South", "AL": "South", "GA": "South", "SC": "South", "NC": "South", "TN": "South", "KY": "South", "FL": "South", "VA": "South", "WV": "South",
    "NY": "Northeast", "PA": "Northeast", "NJ": "Northeast", "MA": "Northeast", "CT": "Northeast", "RI": "Northeast", "VT": "Northeast", "NH": "Northeast", "ME": "Northeast", "DE": "Northeast", "MD": "Northeast", "DC": "Northeast",
    "OH": "Midwest", "IL": "Midwest", "MI": "Midwest", "MN": "Midwest", "WI": "Midwest", "IA": "Midwest", "MO": "Midwest", "IN": "Midwest", "KS": "Midwest", "NE": "Midwest", "SD": "Midwest", "ND": "Midwest"
}

sample_df["Region"] = sample_df["State"].map(region_map)
print(sample_df["Region"].value_counts())

X = sample_df[selected_cols].copy()
y = sample_df["Region"]

cat_cols = X.select_dtypes(include=["str", "object", "bool"]).columns.tolist()
for col in cat_cols:
    X[col] = X[col].astype("category")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model_region = lgb.LGBMClassifier(objective="multiclass", class_weight="balanced", n_estimators=300, learning_rate=0.05, random_state=42)
model_region.fit(X_train, y_train, categorical_feature=cat_cols)

y_pred_region = model_region.predict(X_test)
print(classification_report(y_test, y_pred_region))
print("Macro F1:", f1_score(y_test, y_pred_region, average="macro"))

In [ ]:


# Use your best severity model (weights: {1:2, 2:1, 3:2, 4:6})
lgb.plot_importance(model_final, max_num_features=15, importance_type="gain")
plt.title("What Actually Drives Accident Severity?")
plt.show()